# 分布式训练基础

## 1. 分布式训练的定义与作用

- **基本定义**：分布式训练是将同一个训练任务分配到多个独立计算设备或节点上协同完成。计算资源不限于 GPU，CPU 集群同样属于分布式系统；本次课程主要讨论 GPU 集群上的分布式训练。
- **主要目的**：随着模型参数量、序列长度和训练批量增大，单张 GPU 往往受到显存容量和计算速度限制。多卡分布式训练可以汇总多张 GPU 的计算与显存资源，从而提高训练吞吐量，或支持单卡无法容纳的大模型。

常见的并行方式包括：

1. **数据并行（Data Parallel，DP）**：每个工作设备保存一份完整模型，分别处理不同的数据子集，再同步梯度或更新结果。
2. **流水线并行（Pipeline Parallel，PP）**：按照模型层或模块划分阶段，将不同阶段放到不同 GPU 上；微批次依次经过各阶段，以降低单卡需要保存的模型规模。
3. **张量并行（Tensor Parallel，TP）**：将单层内部的权重矩阵和计算按维度切分到多张 GPU 上，各卡计算局部结果后再通过集合通信完成组合。
4. **3D 并行**：综合使用数据并行、流水线并行和张量并行。实际拓扑可以跨越一个或多个节点，并不要求每台服务器都保存一份完整模型；也可以根据模型规模和硬件条件只组合其中两种方式。

## 2. 传统数据并行（DP）

- **训练流程**：在 PyTorch `DataParallel` 中，主设备保存原始模型，并在前向传播时将模型复制到其他 GPU；输入数据按批量维度分发到各卡，各模型副本分别完成前向计算，输出汇总到主设备。反向传播时，各副本的梯度归并到原始模型，再由优化器更新参数。
- **主要局限**：主设备承担模型、输出汇总和梯度归并等额外工作，显存与通信负载更高，容易成为瓶颈；同时，`DataParallel` 采用单进程、多线程模式，会受到 Python 调度开销影响，并且主要用于单机多卡，扩展能力有限。
- **代码改造**：模型初始化后使用 `torch.nn.DataParallel` 包装，并指定 GPU 列表，其余训练流程与单卡训练基本一致。虽然使用简单，但在正式多卡训练中通常优先选择 DDP。

## 3. 分布式训练的核心术语

- **Group / Process Group（进程组）**：参与集合通信的一组进程。一个任务可以只有一个默认进程组，也可以根据并行策略创建多个子进程组。
- **World Size**：某个进程组中的总进程数。在一进程对应一张 GPU 的常见配置下，它通常等于参与训练的 GPU 总数。
- **Node（节点）**：一台参与分布式训练的服务器；一个节点可以包含多张 GPU。
- **Global Rank**：进程在整个分布式任务中的唯一编号，范围通常为 `0～world_size-1`。
- **Local Rank**：进程在当前节点内部的编号，通常用于绑定本机 GPU。多节点训练时，Global Rank 可以大于本机 GPU 数量，因此应使用 Local Rank 选择本地设备。
- **设备绑定**：常见做法是读取 `LOCAL_RANK`，再调用 `torch.cuda.set_device(local_rank)`，保证每个进程只操作分配给自己的 GPU。

## 4. 分布式数据并行（DDP）

- **核心原理**：DDP 通常为每张 GPU 启动一个独立进程。每个进程持有完整模型副本，读取不同的数据分片，并独立完成前向传播、损失计算和反向传播。反向传播过程中，DDP 通过 AllReduce 同步各进程的参数梯度；随后，每个进程使用相同梯度执行相同的优化器更新，因此各模型副本始终保持一致。
- **相较 DP 的优势**：DDP 不需要由主设备集中汇总全部输出和梯度；多进程方式也避免了单进程多线程带来的 Python 调度瓶颈。它通常具有更好的负载均衡和扩展效率，并支持单机多卡与多机多卡。
- **局限性**：DDP 仍属于数据并行，每张 GPU 通常都要保存完整模型、梯度和优化器状态，因此模型本身仍需能够放入单卡。7B 模型能否单卡微调取决于量化精度、LoRA/QLoRA 配置、序列长度、每卡批量、梯度检查点和实际可用显存，不能仅依据参数量判断。

## 5. AllReduce 梯度同步

- **基本语义**：AllReduce 先对各进程上的张量执行求和、求平均等归约操作，再把结果返回给所有参与进程。DDP 通常用它将各卡的本地梯度变为相同的全局梯度。
- **通信特点**：AllReduce 不需要长期依赖一个中心进程汇总全部梯度，可以更充分地利用多张 GPU 的通信带宽。Ring AllReduce 是一种常见实现，NCCL 也可能根据数据规模和硬件拓扑选择树形或其他算法。
- **最终结果**：当各模型副本的初始参数一致，并使用相同的同步梯度和优化器状态时，每个进程独立执行 `optimizer.step()` 后，模型参数仍然保持一致。

# DDP、LoRA 与 DeepSpeed 实践要点

## 1. DDP 的核心思想

- **分布式策略**：DDP 通常采用“一张 GPU 对应一个进程”的方式。每个进程保存完整模型副本，只处理自己的数据分片，并独立完成前向传播和反向传播。
- **梯度同步**：CUDA 多卡训练通常使用 NCCL 作为通信后端。DDP 在反向传播期间同步各进程的参数梯度，随后各进程分别执行相同的优化器更新。
- **支持范围**：DDP 本身实现的是数据并行，不负责将模型按层或张量切分；但它可以与流水线并行、张量并行等模型并行策略组合使用。
- **LoRA 场景**：DDP 适合配合 LoRA、QLoRA 等参数高效微调方法使用。基础模型仍会参与前向和反向计算，但通常冻结其原始参数，只更新 LoRA 适配器及显式指定的其他可训练模块。

## 2. GPU 租用与环境复用

- **保存自定义镜像**：完成依赖、模型和数据环境配置后，将实例保存为自定义镜像。后续更换 GPU 或重新创建实例时可直接复用，减少重复配置时间。
- **课堂价格记录**：课堂记录中，AutoDL 的 RTX 2080 Ti 12GB 按量价格约为 1.86 元/小时，RTX 3090 约为 3.83 元/小时。价格、库存和区域启动速度会随时间与供需变化，实际租用前应重新核对。
- **按需调整资源**：热门时段可能缺少空闲 GPU。可在环境保存后先关机或降低实例 GPU 数量，需要训练时再扩容，以减少空闲计费。

## 3. DDP 代码实现基础

- **主要依赖**：需要导入 `os`、Transformers 中的 `AutoTokenizer`、`TrainingArguments`、数据整理器等组件，以及 PEFT 中的 `LoraConfig` 和模型包装函数。
- **训练目标**：课堂示例使用两张 GPU 对 Qwen 1.5B 模型进行 LoRA 微调。单卡 16GB 显存是否足够仍取决于模型精度、序列长度、每卡批量、梯度累积、梯度检查点和优化器状态等配置，应以实际显存测量为准。

## 4. DDP 进程与 GPU 绑定

- **问题来源**：容器内部可见的 GPU 编号可能与宿主机物理编号不同。如果进程使用了错误的设备编号，可能出现显存集中到同一张卡、设备不一致或通信初始化失败。
- **正确绑定**：读取 `LOCAL_RANK`，并使用 `torch.cuda.set_device(local_rank)` 将当前进程绑定到对应的本地 GPU。单机两卡时表现为 Local Rank 0 对应容器内 GPU 0，Local Rank 1 对应容器内 GPU 1；多节点训练时不能直接用 Global Rank 充当本地 GPU 编号。
- **QLoRA 加载注意**：一进程一卡的 QLoRA 训练中，应确保当前进程把量化模型加载到自己的本地 GPU；通常不应使用会跨多卡自动切分模型的 `device_map="auto"`，以免与 DDP 的进程—设备映射冲突。

## 5. DDP 训练参数配置

- **移除未使用列**：`remove_unused_columns=True` 会删除模型 `forward()` 不需要的数据字段，例如原始文本或元数据；它不负责删除最后一个不完整批次。若确实需要丢弃无法组成完整 Batch 的剩余样本，应使用 `dataloader_drop_last=True`。
- **梯度检查点**：启用梯度检查点时，可通过 `gradient_checkpointing_kwargs={"use_reentrant": False}` 选择非重入式实现。这里的 `False` 表示不使用重入式实现，并不等于关闭梯度检查点。
- **未使用参数检查**：当确认所有可训练参数都会参与每轮前向与反向传播时，LoRA 训练常设置 `ddp_find_unused_parameters=False`，以避免额外的计算图遍历。但这不是所有 LoRA 模型都必须使用的固定值；如果存在条件分支或确实未参与计算的可训练参数，需要根据模型结构重新判断。

## 6. DeepSpeed ZeRO 的核心思想

- **ZeRO-1**：分片优化器状态。
- **ZeRO-2**：在 ZeRO-1 的基础上继续分片梯度。
- **ZeRO-3**：在 ZeRO-2 的基础上继续分片模型参数。
- **核心收益**：通过消除不同数据并行进程之间的冗余模型状态，降低每张 GPU 的显存占用，使更大的模型或批量能够运行。其代价通常是增加集合通信、参数收集和调度开销，因此本质上是显存与通信/计算时间之间的权衡。
- **通信方式**：ZeRO 会根据阶段使用 AllGather、ReduceScatter 等集合通信。在需要计算某一层时收集所需参数，在梯度归约后仅保留本进程负责的分片。

## 7. DeepSpeed 与模型加载顺序

- **适用条件**：当使用 Transformers 的 ZeRO-3 分片加载时，需要在 `AutoModel.from_pretrained()` 之前创建包含 DeepSpeed 配置的 `TrainingArguments`；不使用 Trainer 时，则应提前创建并保持 `HfDeepSpeedConfig`。
- **原因**：只有在模型加载前激活 ZeRO-3，Transformers 才能在 `from_pretrained()` 阶段识别参数分片策略，避免先将完整模型加载到单张 GPU。该顺序要求主要针对 ZeRO-3 的分片加载，不应泛化为所有 DeepSpeed/ZeRO 配置都必须如此。

## 8. DeepSpeed 配置文件

- **配置方式**：可以在 Accelerate 的 YAML 文件中直接描述 DeepSpeed 设置，也可以单独编写 DeepSpeed JSON 文件并在 YAML 中引用；使用 Transformers Trainer 时，也可以通过 `TrainingArguments(deepspeed=...)` 指定配置。
- **精度一致性**：模型加载、`TrainingArguments`、Accelerate 配置和 DeepSpeed JSON 中的 FP16/BF16 设置应保持一致，避免精度配置冲突。
- **ZeRO-3 与 Offload**：ZeRO-3 本身已经能够分片模型参数，不要求必须开启 CPU Offload。显存不足时，可以额外将优化器状态或模型参数卸载到 CPU/NVMe；这样会进一步节省显存，但也会增加主机内存、数据传输和训练时间开销。

## 9. DeepSpeed 启动方式

- **Accelerate 启动**：本课程使用 `accelerate launch --config_file 配置文件路径 训练脚本路径` 启动多进程训练，而不是直接执行 `python 训练脚本.py`。
- **共享内存问题**：容器的 `/dev/shm` 过小时，DataLoader、多进程通信或 NCCL 任务可能失败。个人租用实例时可在创建容器时增大共享内存；共享教学容器若无相应权限，需要联系管理员或调整数据加载配置。